# Kev-0.8B LoRA fine-tuning — LANESHIFT runbook

This notebook is the controlled Colab training path for LANESHIFT. It fine-tunes
**`jaredpalmer/kev-0.8b@v1.0`** with a LoRA delta; it does not train the bare
Qwen base from scratch.

- Inputs: a Kev-format labelled `train.jsonl` and `validation.jsonl`.
- Outputs: the adapter/head checkpoint on Google Drive, never in Git.
- The notebook runs one training job and exits; it does not start a persistent
  API.
- The released model weights, tokenizer and external `.data` blobs stay outside
  GitHub. See `docs/MODELS.md`.


## 1. GPU runtime

Use **Runtime > Change runtime type > T4 GPU** for the free path, or L4/A100 if
available. The Kev repository requires Python 3.12-3.13. Python 3.12+ is
available on current Colab images; switch runtimes if it is older.


In [ ]:
!nvidia-smi
import sys, torch
print("python", sys.version)
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("bf16_supported", torch.cuda.is_bf16_supported())


## 2. Install Kev at the pinned commit

The project was developed against this upstream commit. Installing at a moving
tip makes the checkpoint/config compatibility test meaningless.


In [ ]:
!git clone https://github.com/jaredpalmer/kev.git /content/kev
!git -C /content/kev checkout 2074cc44256200bb37540103fa4572901faa10f3
%cd /content/kev
%pip install -q -e .


## 3. Mount Drive and upload the labelled data

The Kev training format is one JSON object per line:

```json
{"state":"...","questions":{"next_action":{"type":"choice","instructions":"...","criteria":{"LEFT":"...","STAY":"..."},"label":"STAY"}}}
```

The LANESHIFT pilot dataset in `datasets/pilot-v1/` is the 1,348-row decision
dataset; a Kev-format export must be produced from the recorded decisions by
the project exporter before training. Upload `train.jsonl` and
`validation.jsonl` here.


In [ ]:
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
uploaded = files.upload()
for name, data in uploaded.items():
    Path('/content', name).write_bytes(data)
print("train exists", Path('/content/train.jsonl').is_file())
print("validation exists", Path('/content/validation.jsonl').is_file())


## 4. Train

Free T4 does not support native bf16; use fp32. On L4/A100 switch
`--dtype bf16` and keep `--weights_dtype fp32` to match the released
checkpoint. The first run is one epoch with accumulation; focal loss is for the
strong STAY imbalance in the rule-generated pilot data. If you use the natural
distribution instead, drop `--focal_gamma`.


In [ ]:
!python -m kev.train \
  --data /content/train.jsonl \
  --base Qwen/Qwen3.5-0.8B-Base \
  --base_revision dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68 \
  --init_from jaredpalmer/kev-0.8b@v1.0 \
  --epochs 1 --lr 2e-5 --batch 1 --accum 8 \
  --dtype fp32 --weights_dtype fp32 --checkpointing 1 \
  --focal_gamma 1.5 --device cuda --seed 0 \
  --out /content/drive/MyDrive/laneshift-kev-r1


## 5. Evaluate and calibrate

Score the untouched checkpoint and the fine-tuned run on the same validation
file. Fit a temperature on validation before reporting Brier/log loss/ECE.
Accuracy is unchanged by temperature; calibration is not.


In [ ]:
!python -m kev.benchmark --run jaredpalmer/kev-0.8b@v1.0 --data /content/validation.jsonl --out /content/drive/MyDrive/laneshift-kev-eval-original --device cuda
!python -m kev.benchmark --run /content/drive/MyDrive/laneshift-kev-r1 --data /content/validation.jsonl --out /content/drive/MyDrive/laneshift-kev-eval-ft --device cuda


## 6. Export for LANESHIFT

The ONNX export source is in `tools/kev-onnx-export/`. For a fine-tuned run,
write a **new** external-data blob from the merged fine-tuned state dict. Do not
use the reuse path that points at the original `kev07b_fp16.data`; a
same-shape state dict would silently export the original weights.

The ONNX graph, new `.data` blob and tokenizer are external artifacts. Store
them on Drive or Hugging Face Hub; do not commit them.
